In [ ]:
# ============================================================
# MASK R-CNN for DeepFashion-style Detection + Segmentation
# Fully corrected Kaggle notebook
#
# Fixes included:
# 1) Proper COCO-pretrained Mask R-CNN loading with head replacement
#    so num_classes != 91 works correctly
# 2) Robust recursive indexing for test images / JSON files
#    so test split does not end up with 0 samples unnecessarily
#
# Outputs:
# - best checkpoint
# - last checkpoint
# - history
# - plots
# - validation + test predictions
# - per-class segmentation / detection metrics
# - overlay images
# ============================================================

import os
import gc
import json
import math
import glob
import random
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image, ImageDraw

import torch
import torchvision
from torch.utils.data import Dataset, DataLoader, Subset
from torchvision.transforms import functional as TF
from torchvision.ops import box_iou

from torchvision.models import ResNet50_Weights
from torchvision.models.detection import (
    maskrcnn_resnet50_fpn_v2,
    MaskRCNN_ResNet50_FPN_V2_Weights,
)
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection.mask_rcnn import MaskRCNNPredictor

from torch.amp import autocast, GradScaler
from tqdm.auto import tqdm

from sklearn.metrics import roc_curve, roc_auc_score

# ============================================================
# 1. Reproducibility
# ============================================================
SEED = 42

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)

# ============================================================
# 2. Device / performance
# ============================================================
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if DEVICE.type != "cuda":
    raise RuntimeError("GPU not enabled. In Kaggle, set Accelerator = GPU and rerun.")

AMP_ENABLED = True
AMP_DEVICE = "cuda"

print("Using device:", DEVICE)
print("Torch:", torch.__version__)
print("Torchvision:", torchvision.__version__)

torch.backends.cudnn.benchmark = True
try:
    torch.set_float32_matmul_precision("high")
except Exception:
    pass

SCALER = GradScaler(device="cuda", enabled=AMP_ENABLED)

# ============================================================
# 3. Config
# ============================================================
OUT_DIR = "/kaggle/working/maskrcnn_outputs_v2"
os.makedirs(OUT_DIR, exist_ok=True)

# ---- Runtime / training budget ----
MAX_EPOCHS = 12                 # set 15 for full run
BATCH_SIZE = 1
GRAD_ACCUM_STEPS = 2
NUM_WORKERS = min(4, os.cpu_count() or 2)
LR = 2e-4
WEIGHT_DECAY = 1e-4
PATIENCE = 4
EVAL_EVERY = 2
VAL_SUBSET_IMAGES = 2000        # set larger like 2000 for fuller quick validation

# ---- Image size ----
MIN_SIZE = 512
MAX_SIZE = 640

# ---- Detection ----
BOX_DETECTIONS_PER_IMG = 30
TRAINABLE_BACKBONE_LAYERS = 3

# ---- Thresholds ----
AP_MIN_SCORE = 0.05
SAVE_SCORE_THRESH = 0.50
MATCH_IOU_THRESH = 0.50

# ---- Output controls ----
SAVE_SAMPLE_OVERLAYS = 100      # increase later if needed
SAVE_ALL_INSTANCE_MAPS = True

TRAIN_LIMIT = None
VAL_LIMIT = None
TEST_LIMIT = None

# ============================================================
# 4. Category mapping
# ============================================================
ORIG_ID_TO_NAME = {
    1: "short sleeve top",
    2: "long sleeve top",
    3: "short sleeve outwear",
    4: "long sleeve outwear",
    5: "vest",
    6: "sling",
    7: "shorts",
    8: "trousers",
    9: "skirt",
    10: "short sleeve dress",
    11: "long sleeve dress",
    12: "vest dress",
    13: "sling dress",
}

# ============================================================
# 5. Path resolution
# ============================================================
def first_existing_path(candidates):
    for p in candidates:
        if p and os.path.exists(p):
            return p
    return None

def first_glob_match(patterns):
    for pat in patterns:
        matches = glob.glob(pat, recursive=True)
        if matches:
            return matches[0]
    return None

def resolve_dir(name, candidates, glob_patterns=None, required=True):
    path = first_existing_path(candidates)
    if path is None and glob_patterns:
        path = first_glob_match(glob_patterns)
    if path is None and required:
        raise FileNotFoundError(f"Could not resolve path for {name}")
    return path

train_image_dir = resolve_dir(
    "train_image_dir",
    candidates=[
        "/kaggle/input/training_50k/train_50k/images",
        "/kaggle/input/training-50k/train_50k/images",
        "/kaggle/input/training_50k/deepfashion_50k/images",
        "/kaggle/input/datasets/varun000reddy/training_50k/train_50k/images",
        "/kaggle/input/datasets/varun000reddy/training-50k/train_50k/images",
    ],
    glob_patterns=[
        "/kaggle/input/**/train_50k/images",
        "/kaggle/input/**/deepfashion_50k/images",
    ],
)

train_anno_dir = resolve_dir(
    "train_anno_dir",
    candidates=[
        "/kaggle/input/training_50k/train_50k/annos",
        "/kaggle/input/training-50k/train_50k/annos",
        "/kaggle/input/training_50k/deepfashion_50k/annos",
        "/kaggle/input/datasets/varun000reddy/training_50k/train_50k/annos",
        "/kaggle/input/datasets/varun000reddy/training-50k/train_50k/annos",
    ],
    glob_patterns=[
        "/kaggle/input/**/train_50k/annos",
        "/kaggle/input/**/deepfashion_50k/annos",
    ],
)

val_image_dir = resolve_dir(
    "val_image_dir",
    candidates=[
        "/kaggle/input/validation/validation/image",
        "/kaggle/input/datasets/varun000reddy/validation/validation/image",
    ],
    glob_patterns=[
        "/kaggle/input/**/validation/image",
    ],
)

val_anno_dir = resolve_dir(
    "val_anno_dir",
    candidates=[
        "/kaggle/input/validation/validation/annos",
        "/kaggle/input/datasets/varun000reddy/validation/validation/annos",
    ],
    glob_patterns=[
        "/kaggle/input/**/validation/annos",
    ],
)

test_image_dir = resolve_dir(
    "test_image_dir",
    candidates=[
        "/kaggle/input/testing/test/test/image",
        "/kaggle/input/testing/test/image",
        "/kaggle/input/datasets/varun000reddy/testing/test/test/image",
        "/kaggle/input/datasets/varun000reddy/testing/test/image",
    ],
    glob_patterns=[
        "/kaggle/input/**/test/test/image",
        "/kaggle/input/**/test/image",
    ],
    required=False,
)

test_anno_dir = resolve_dir(
    "test_anno_dir",
    candidates=[
        "/kaggle/input/testing/test/json_for_test",
        "/kaggle/input/testing/json_for_test",
        "/kaggle/input/datasets/varun000reddy/testing/test/json_for_test",
        "/kaggle/input/datasets/varun000reddy/testing/json_for_test",
    ],
    glob_patterns=[
        "/kaggle/input/**/json_for_test",
    ],
    required=False,
)

print("Train image dir:", train_image_dir)
print("Train anno dir :", train_anno_dir)
print("Val image dir  :", val_image_dir)
print("Val anno dir   :", val_anno_dir)
print("Test image dir :", test_image_dir)
print("Test anno dir  :", test_anno_dir)

# ============================================================
# 6. Recursive file utilities
# ============================================================
IMAGE_EXTS = [".jpg", ".jpeg", ".png", ".bmp", ".webp"]
JSON_EXTS = [".json"]

def recursive_list_files(root_dir, exts):
    if root_dir is None or not os.path.exists(root_dir):
        return []
    files = []
    for ext in exts:
        files.extend(glob.glob(os.path.join(root_dir, "**", f"*{ext}"), recursive=True))
        files.extend(glob.glob(os.path.join(root_dir, "**", f"*{ext.upper()}"), recursive=True))
    files = sorted(list(set(files)))
    return files

def recursive_image_files(root_dir, limit=None):
    files = recursive_list_files(root_dir, IMAGE_EXTS)
    if limit is not None:
        files = files[:limit]
    return files

def recursive_json_files(root_dir):
    return recursive_list_files(root_dir, JSON_EXTS)

def build_stem_to_path_map(file_list):
    stem_to_path = {}
    duplicates = 0
    for p in file_list:
        stem = Path(p).stem
        if stem in stem_to_path:
            duplicates += 1
            # keep first occurrence
            continue
        stem_to_path[stem] = p
    return stem_to_path, duplicates

def clamp(v, lo, hi):
    return max(lo, min(v, hi))

def parse_bbox(raw_bbox, width, height):
    if raw_bbox is None:
        return None

    if isinstance(raw_bbox, dict):
        vals = list(raw_bbox.values())
    else:
        vals = list(raw_bbox)

    if len(vals) != 4:
        return None

    x1, y1, x2, y2 = map(float, vals)

    # Convert [x, y, w, h] if needed
    if x2 <= x1 or y2 <= y1:
        if vals[2] > 0 and vals[3] > 0:
            x2 = x1 + vals[2]
            y2 = y1 + vals[3]

    x1 = clamp(x1, 0, width - 1)
    y1 = clamp(y1, 0, height - 1)
    x2 = clamp(x2, 0, width - 1)
    y2 = clamp(y2, 0, height - 1)

    if x2 <= x1 or y2 <= y1:
        return None

    return [x1, y1, x2, y2]

def segmentation_to_polygons(segmentation):
    polygons = []

    if segmentation is None:
        return polygons

    if isinstance(segmentation, dict):
        for v in segmentation.values():
            polygons.extend(segmentation_to_polygons(v))
        return polygons

    if isinstance(segmentation, (list, tuple)):
        if len(segmentation) == 0:
            return polygons

        if all(isinstance(v, (int, float)) for v in segmentation):
            if len(segmentation) >= 6 and len(segmentation) % 2 == 0:
                polygons.append(list(map(float, segmentation)))
            return polygons

        if all(isinstance(v, (list, tuple)) and len(v) == 2 for v in segmentation):
            flat = []
            for x, y in segmentation:
                flat.extend([float(x), float(y)])
            if len(flat) >= 6:
                polygons.append(flat)
            return polygons

        for item in segmentation:
            polygons.extend(segmentation_to_polygons(item))
        return polygons

    return polygons

def polygons_to_mask(polygons, width, height):
    mask_img = Image.new("L", (width, height), 0)
    draw = ImageDraw.Draw(mask_img)
    drawn = False

    for poly in polygons:
        if len(poly) < 6 or len(poly) % 2 != 0:
            continue
        xy = [(poly[i], poly[i + 1]) for i in range(0, len(poly), 2)]
        draw.polygon(xy, outline=1, fill=1)
        drawn = True

    mask = np.array(mask_img, dtype=np.uint8)
    return mask, drawn

def bbox_to_mask(box, width, height):
    x1, y1, x2, y2 = map(int, box)
    mask = np.zeros((height, width), dtype=np.uint8)
    mask[y1:y2 + 1, x1:x2 + 1] = 1
    return mask

def make_empty_target(height, width, idx):
    return {
        "boxes": torch.zeros((0, 4), dtype=torch.float32),
        "labels": torch.zeros((0,), dtype=torch.int64),
        "masks": torch.zeros((0, height, width), dtype=torch.uint8),
        "image_id": torch.tensor([idx]),
        "area": torch.zeros((0,), dtype=torch.float32),
        "iscrowd": torch.zeros((0,), dtype=torch.int64),
    }

# ============================================================
# 7. Top-5 category discovery from TRAIN split
# ============================================================
def compute_top5_categories(train_anno_dir, cache_path):
    if os.path.exists(cache_path):
        with open(cache_path, "r") as f:
            obj = json.load(f)
        top5 = obj["top5"]
        counts = {int(k): int(v) for k, v in obj["counts"].items()}
        return top5, counts

    counts = Counter()
    json_files = recursive_json_files(train_anno_dir)

    for anno_path in tqdm(json_files, desc="Counting category frequencies"):
        with open(anno_path, "r") as f:
            data = json.load(f)

        for key, value in data.items():
            if key.startswith("item") and isinstance(value, dict):
                cat_id = value.get("category_id", None)
                if cat_id in ORIG_ID_TO_NAME:
                    counts[cat_id] += 1

    top5 = [cat_id for cat_id, _ in counts.most_common(5)]

    with open(cache_path, "w") as f:
        json.dump(
            {
                "top5": top5,
                "counts": {str(k): int(v) for k, v in counts.items()},
            },
            f,
            indent=2
        )

    return top5, dict(counts)

TOP5_CACHE = os.path.join(OUT_DIR, "top5_categories.json")
TOP5_ORIG_IDS, CATEGORY_COUNTS = compute_top5_categories(train_anno_dir, TOP5_CACHE)

ORIG_TO_TRAIN_LABEL = {orig_id: i + 1 for i, orig_id in enumerate(TOP5_ORIG_IDS)}
TRAIN_LABEL_TO_NAME = {0: "background"}
for orig_id, train_id in ORIG_TO_TRAIN_LABEL.items():
    TRAIN_LABEL_TO_NAME[train_id] = ORIG_ID_TO_NAME[orig_id]

NUM_CLASSES = 1 + len(TOP5_ORIG_IDS)
CLASS_IDS = sorted(list(ORIG_TO_TRAIN_LABEL.values()))

print("Top-5 original category IDs:", TOP5_ORIG_IDS)
print("Top-5 names:", [ORIG_ID_TO_NAME[x] for x in TOP5_ORIG_IDS])
print("Train label mapping:", ORIG_TO_TRAIN_LABEL)

# ============================================================
# 8. Dataset
# ============================================================
class FashionMaskRCNNDataset(Dataset):
    def __init__(
        self,
        image_dir,
        anno_dir=None,
        allowed_orig_ids=None,
        orig_to_train_label=None,
        limit=None,
        is_train=False,
        cache_name="index_cache.json",
        fallback_to_image_only=True,
    ):
        self.image_dir = image_dir
        self.anno_dir = anno_dir
        self.allowed_orig_ids = set(allowed_orig_ids) if allowed_orig_ids is not None else None
        self.orig_to_train_label = orig_to_train_label
        self.limit = limit
        self.is_train = is_train
        self.cache_path = os.path.join(OUT_DIR, cache_name)
        self.fallback_to_image_only = fallback_to_image_only

        self.has_annotation_source = bool(anno_dir is not None and os.path.exists(anno_dir))
        self.matched_json_count = 0
        self.uses_image_only_fallback = False

        self.samples = self._build_or_load_index()

    def _image_only_samples(self):
        img_files = recursive_image_files(self.image_dir, limit=self.limit)
        samples = []
        for img_path in tqdm(img_files, desc=f"Indexing images only: {os.path.basename(self.image_dir)}"):
            stem = Path(img_path).stem
            samples.append({
                "image_path": img_path,
                "stem": stem,
                "instances": [],
                "has_gt": False,
            })
        return samples

    def _build_or_load_index(self):
        if os.path.exists(self.cache_path):
            with open(self.cache_path, "r") as f:
                payload = json.load(f)

            if isinstance(payload, dict) and "samples" in payload:
                samples = payload["samples"]
                self.matched_json_count = int(payload.get("matched_json_count", 0))
                self.uses_image_only_fallback = bool(payload.get("uses_image_only_fallback", False))
            else:
                samples = payload
                self.matched_json_count = 0
                self.uses_image_only_fallback = False

            if self.limit is not None:
                samples = samples[:self.limit]

            print(f"Loaded cache {self.cache_path} with {len(samples)} samples")
            return samples

        # image-only dataset if no annotations
        if not self.has_annotation_source:
            samples = self._image_only_samples()
            self.uses_image_only_fallback = True

            payload = {
                "samples": samples,
                "matched_json_count": 0,
                "uses_image_only_fallback": True,
            }
            with open(self.cache_path, "w") as f:
                json.dump(payload, f)

            print(f"Built cache {self.cache_path} with {len(samples)} image-only samples")
            return samples

        image_files = recursive_image_files(self.image_dir)
        json_files = recursive_json_files(self.anno_dir)

        stem_to_img, dup_count = build_stem_to_path_map(image_files)
        print(f"Found {len(image_files)} image files ({dup_count} duplicate stems skipped) in {self.image_dir}")
        print(f"Found {len(json_files)} json files in {self.anno_dir}")

        samples = []
        matched = 0

        for anno_path in tqdm(json_files, desc=f"Indexing {os.path.basename(self.anno_dir)}"):
            stem = Path(anno_path).stem
            img_path = stem_to_img.get(stem, None)

            if img_path is None:
                continue

            with open(anno_path, "r") as f:
                data = json.load(f)

            instances = []
            for key, value in data.items():
                if not key.startswith("item"):
                    continue
                if not isinstance(value, dict):
                    continue

                orig_cat_id = value.get("category_id", None)
                if self.allowed_orig_ids is not None and orig_cat_id not in self.allowed_orig_ids:
                    continue

                instances.append({
                    "orig_cat_id": orig_cat_id,
                    "bounding_box": value.get("bounding_box", None),
                    "segmentation": value.get("segmentation", None),
                })

            samples.append({
                "image_path": img_path,
                "stem": stem,
                "instances": instances,
                "has_gt": True,
            })

            matched += 1
            if self.limit is not None and len(samples) >= self.limit:
                break

        self.matched_json_count = matched

        # If no matches, or somehow samples ended up 0, fallback to image-only inference indexing
        if len(samples) == 0 and self.fallback_to_image_only:
            print("Warning: No matched annotation-image pairs found. Falling back to image-only indexing.")
            samples = self._image_only_samples()
            self.uses_image_only_fallback = True
        else:
            self.uses_image_only_fallback = False

        payload = {
            "samples": samples,
            "matched_json_count": int(self.matched_json_count),
            "uses_image_only_fallback": bool(self.uses_image_only_fallback),
        }

        with open(self.cache_path, "w") as f:
            json.dump(payload, f)

        print(f"Built cache {self.cache_path} with {len(samples)} samples")
        print(f"Matched annotation-image pairs: {self.matched_json_count}")
        print(f"Image-only fallback used: {self.uses_image_only_fallback}")
        return samples

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sample = self.samples[idx]
        image = Image.open(sample["image_path"]).convert("RGB")
        width, height = image.size

        boxes = []
        labels = []
        masks = []

        for inst in sample["instances"]:
            orig_cat_id = inst.get("orig_cat_id", None)

            if self.allowed_orig_ids is not None and orig_cat_id not in self.allowed_orig_ids:
                continue

            box = parse_bbox(inst.get("bounding_box", None), width, height)
            if box is None:
                continue

            polygons = segmentation_to_polygons(inst.get("segmentation", None))
            mask, ok = polygons_to_mask(polygons, width, height)
            if not ok:
                mask = bbox_to_mask(box, width, height)

            if mask.sum() == 0:
                continue

            boxes.append(box)
            if self.orig_to_train_label is not None and orig_cat_id in self.orig_to_train_label:
                labels.append(self.orig_to_train_label[orig_cat_id])
            else:
                labels.append(0)
            masks.append(mask)

        if len(boxes) == 0:
            target = make_empty_target(height, width, idx)
        else:
            boxes = torch.tensor(boxes, dtype=torch.float32)
            labels = torch.tensor(labels, dtype=torch.int64)
            masks = torch.tensor(np.stack(masks), dtype=torch.uint8)
            area = (boxes[:, 2] - boxes[:, 0]) * (boxes[:, 3] - boxes[:, 1])
            iscrowd = torch.zeros((len(labels),), dtype=torch.int64)

            target = {
                "boxes": boxes,
                "labels": labels,
                "masks": masks,
                "image_id": torch.tensor([idx]),
                "area": area,
                "iscrowd": iscrowd,
            }

        # light training augmentation
        if self.is_train and random.random() < 0.5:
            image = TF.hflip(image)
            if len(target["boxes"]) > 0:
                boxes = target["boxes"].clone()
                boxes[:, [0, 2]] = width - boxes[:, [2, 0]]
                target["boxes"] = boxes
                target["masks"] = torch.flip(target["masks"], dims=[2])

        image = TF.to_tensor(image)

        meta = {
            "stem": sample["stem"],
            "image_path": sample["image_path"],
            "height": height,
            "width": width,
            "has_gt": bool(sample.get("has_gt", False)),
        }

        return image, target, meta

def collate_fn(batch):
    images, targets, metas = zip(*batch)
    return list(images), list(targets), list(metas)

# ============================================================
# 9. Build datasets
# ============================================================
top5_tag = "_".join(map(str, TOP5_ORIG_IDS))

train_dataset = FashionMaskRCNNDataset(
    image_dir=train_image_dir,
    anno_dir=train_anno_dir,
    allowed_orig_ids=TOP5_ORIG_IDS,
    orig_to_train_label=ORIG_TO_TRAIN_LABEL,
    limit=TRAIN_LIMIT,
    is_train=True,
    cache_name=f"train_index_{top5_tag}.json",
)

val_dataset = FashionMaskRCNNDataset(
    image_dir=val_image_dir,
    anno_dir=val_anno_dir,
    allowed_orig_ids=TOP5_ORIG_IDS,
    orig_to_train_label=ORIG_TO_TRAIN_LABEL,
    limit=VAL_LIMIT,
    is_train=False,
    cache_name=f"val_index_{top5_tag}.json",
)

if test_image_dir is not None:
    test_dataset = FashionMaskRCNNDataset(
        image_dir=test_image_dir,
        anno_dir=test_anno_dir,
        allowed_orig_ids=TOP5_ORIG_IDS if test_anno_dir is not None else None,
        orig_to_train_label=ORIG_TO_TRAIN_LABEL if test_anno_dir is not None else None,
        limit=TEST_LIMIT,
        is_train=False,
        cache_name=f"test_index_{top5_tag}.json",
    )
else:
    test_dataset = None

print("Train samples:", len(train_dataset))
print("Val samples  :", len(val_dataset))
print("Test samples :", 0 if test_dataset is None else len(test_dataset))

if test_dataset is not None:
    print("Test matched annotation-image pairs:", test_dataset.matched_json_count)
    print("Test image-only fallback used     :", test_dataset.uses_image_only_fallback)

# ============================================================
# 10. DataLoaders
# ============================================================
loader_common_kwargs = {
    "batch_size": BATCH_SIZE,
    "num_workers": NUM_WORKERS,
    "collate_fn": collate_fn,
    "pin_memory": True,
    "persistent_workers": NUM_WORKERS > 0,
}
if NUM_WORKERS > 0:
    loader_common_kwargs["prefetch_factor"] = 2

train_loader = DataLoader(train_dataset, shuffle=True, **loader_common_kwargs)
val_loader = DataLoader(val_dataset, shuffle=False, **loader_common_kwargs)

subset_size = min(VAL_SUBSET_IMAGES, len(val_dataset))
rng = np.random.default_rng(SEED)
subset_indices = rng.choice(len(val_dataset), size=subset_size, replace=False).tolist()
val_subset = Subset(val_dataset, subset_indices)
val_subset_loader = DataLoader(val_subset, shuffle=False, **loader_common_kwargs)

if test_dataset is not None and len(test_dataset) > 0:
    test_loader = DataLoader(test_dataset, shuffle=False, **loader_common_kwargs)
else:
    test_loader = None

# ============================================================
# 11. Ground-truth sanity visualization
# ============================================================
def draw_gt_sample(dataset, save_path, idx=0):
    image, target, meta = dataset[idx]
    img_np = image.permute(1, 2, 0).numpy()

    fig, ax = plt.subplots(1, 1, figsize=(10, 10))
    ax.imshow(img_np)
    ax.set_title(f"Ground Truth Sample: {meta['stem']}")
    ax.axis("off")

    cmap = plt.get_cmap("tab10")

    for i in range(len(target["boxes"])):
        x1, y1, x2, y2 = target["boxes"][i].tolist()
        label = int(target["labels"][i].item())
        mask = target["masks"][i].numpy()

        color = cmap(label % 10)

        overlay = np.zeros((mask.shape[0], mask.shape[1], 4), dtype=np.float32)
        overlay[..., 0] = color[0]
        overlay[..., 1] = color[1]
        overlay[..., 2] = color[2]
        overlay[..., 3] = mask * 0.35
        ax.imshow(overlay)

        rect = plt.Rectangle(
            (x1, y1), x2 - x1, y2 - y1,
            fill=False, edgecolor=color[:3], linewidth=2
        )
        ax.add_patch(rect)

        ax.text(
            x1,
            max(0, y1 - 5),
            TRAIN_LABEL_TO_NAME.get(label, f"class_{label}"),
            color="white",
            fontsize=10,
            bbox=dict(facecolor=color[:3], alpha=0.8, pad=2)
        )

    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.close(fig)

gt_path = os.path.join(OUT_DIR, "ground_truth_example.png")
draw_gt_sample(train_dataset, gt_path, idx=0)
print("Saved GT example:", gt_path)

# ============================================================
# 12. Model
# ============================================================
def get_model(num_classes):
    """
    Correct fix:
    1) Load COCO-pretrained Mask R-CNN
    2) Replace the box and mask heads with dataset-specific heads
    """

    try:
        model = maskrcnn_resnet50_fpn_v2(
            weights=MaskRCNN_ResNet50_FPN_V2_Weights.DEFAULT,
            min_size=MIN_SIZE,
            max_size=MAX_SIZE,
            box_detections_per_img=BOX_DETECTIONS_PER_IMG,
            trainable_backbone_layers=TRAINABLE_BACKBONE_LAYERS,
        )

        # Replace box classifier head
        in_features_box = model.roi_heads.box_predictor.cls_score.in_features
        model.roi_heads.box_predictor = FastRCNNPredictor(
            in_features_box,
            num_classes
        )

        # Replace mask predictor head
        in_features_mask = model.roi_heads.mask_predictor.conv5_mask.in_channels
        hidden_layer = 256
        model.roi_heads.mask_predictor = MaskRCNNPredictor(
            in_features_mask,
            hidden_layer,
            num_classes
        )

        print("Loaded COCO-pretrained Mask R-CNN and replaced heads successfully.")
        return model

    except Exception as e:
        print("Full COCO-pretrained weights could not be loaded.")
        print("Falling back to ImageNet-pretrained ResNet-50 backbone only.")
        print("Reason:", str(e))

        model = maskrcnn_resnet50_fpn_v2(
            weights=None,
            weights_backbone=ResNet50_Weights.IMAGENET1K_V2,
            num_classes=num_classes,
            min_size=MIN_SIZE,
            max_size=MAX_SIZE,
            box_detections_per_img=BOX_DETECTIONS_PER_IMG,
            trainable_backbone_layers=TRAINABLE_BACKBONE_LAYERS,
        )
        return model

model = get_model(NUM_CLASSES).to(DEVICE)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LR,
    weight_decay=WEIGHT_DECAY,
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=MAX_EPOCHS,
    eta_min=LR * 0.1,
)

print("Model created.")

# ============================================================
# 13. Helpers
# ============================================================
def move_targets_to_device(targets, device):
    moved = []
    for t in targets:
        moved.append({k: v.to(device, non_blocking=True) for k, v in t.items()})
    return moved

def compute_ap_101pt(preds, gt_by_image, iou_thr=0.5):
    npos = sum(len(v) for v in gt_by_image.values())
    if npos == 0:
        return np.nan

    preds = sorted(preds, key=lambda x: x["score"], reverse=True)
    matched = {img_id: np.zeros(len(boxes), dtype=bool) for img_id, boxes in gt_by_image.items()}

    tp = np.zeros(len(preds), dtype=np.float32)
    fp = np.zeros(len(preds), dtype=np.float32)

    for i, pred in enumerate(preds):
        image_id = pred["image_id"]
        pred_box = torch.tensor(pred["box"], dtype=torch.float32).unsqueeze(0)

        gt_boxes = gt_by_image.get(image_id, [])
        if len(gt_boxes) == 0:
            fp[i] = 1.0
            continue

        gt_boxes_tensor = torch.tensor(np.array(gt_boxes), dtype=torch.float32)
        ious = box_iou(pred_box, gt_boxes_tensor).squeeze(0).numpy()

        best_idx = int(np.argmax(ious))
        best_iou = float(ious[best_idx])

        if best_iou >= iou_thr and not matched[image_id][best_idx]:
            tp[i] = 1.0
            matched[image_id][best_idx] = True
        else:
            fp[i] = 1.0

    tp_cum = np.cumsum(tp)
    fp_cum = np.cumsum(fp)

    recalls = tp_cum / (npos + 1e-12)
    precisions = tp_cum / (tp_cum + fp_cum + 1e-12)

    ap = 0.0
    for t in np.linspace(0, 1, 101):
        if np.any(recalls >= t):
            p = np.max(precisions[recalls >= t])
        else:
            p = 0.0
        ap += p / 101.0

    return ap

def safe_mean(values):
    vals = [v for v in values if v is not None and not np.isnan(v)]
    if len(vals) == 0:
        return np.nan
    return float(np.mean(vals))

def save_overlay_image(image_tensor, boxes, labels, scores, masks, save_path):
    palette = [
        (230, 25, 75), (60, 180, 75), (255, 225, 25), (0, 130, 200),
        (245, 130, 48), (145, 30, 180), (70, 240, 240), (240, 50, 230),
        (210, 245, 60), (250, 190, 190)
    ]

    img = TF.to_pil_image(image_tensor).convert("RGBA")
    overlay = Image.new("RGBA", img.size, (0, 0, 0, 0))

    for box, label, score, mask in zip(boxes, labels, scores, masks):
        color = palette[int(label) % len(palette)]
        rgba = color + (90,)

        mask_img = Image.fromarray((mask.astype(np.uint8) * 255), mode="L")
        color_layer = Image.new("RGBA", img.size, rgba)
        overlay = Image.composite(color_layer, overlay, mask_img)

        draw = ImageDraw.Draw(overlay)
        x1, y1, x2, y2 = [float(v) for v in box]
        draw.rectangle([x1, y1, x2, y2], outline=color + (255,), width=3)

        class_name = TRAIN_LABEL_TO_NAME.get(int(label), f"class_{label}")
        text = f"{class_name} {float(score):.2f}"
        draw.text((x1, max(0, y1 - 12)), text, fill=(255, 255, 255, 255))

    out = Image.alpha_composite(img, overlay).convert("RGB")
    out.save(save_path, quality=90)

# ============================================================
# 14. Training loop
# ============================================================
def train_one_epoch(model, loader, optimizer, device, epoch, scaler, accum_steps=1):
    model.train()
    total_loss = 0.0
    valid_steps = 0
    skipped_steps = 0
    loss_parts_sum = defaultdict(float)

    optimizer.zero_grad(set_to_none=True)

    pbar = tqdm(loader, desc=f"Train Epoch {epoch}")
    for step, (images, targets, metas) in enumerate(pbar, start=1):
        images = [img.to(device, non_blocking=True) for img in images]
        targets = move_targets_to_device(targets, device)

        with autocast(device_type=AMP_DEVICE, enabled=AMP_ENABLED):
            loss_dict = model(images, targets)
            losses = sum(loss for loss in loss_dict.values())
            scaled_loss = losses / accum_steps

        if not torch.isfinite(losses):
            skipped_steps += 1
            optimizer.zero_grad(set_to_none=True)
            print(f"[Epoch {epoch} | Step {step}] Non-finite loss. Skipping batch.")
            continue

        scaler.scale(scaled_loss).backward()

        if step % accum_steps == 0 or step == len(loader):
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)

        total_loss += float(losses.item())
        valid_steps += 1

        for k, v in loss_dict.items():
            if torch.isfinite(v):
                loss_parts_sum[k] += float(v.item())

        avg_loss = total_loss / max(valid_steps, 1)
        pbar.set_postfix(loss=f"{avg_loss:.4f}", skipped=skipped_steps)

    if valid_steps == 0:
        raise RuntimeError("All training batches were invalid. Training stopped.")

    loss_parts_avg = {k: v / valid_steps for k, v in loss_parts_sum.items()}
    return total_loss / valid_steps, loss_parts_avg

# ============================================================
# 15. Evaluation
# ============================================================
def evaluate_split(
    model,
    loader,
    device,
    split_name,
    out_dir,
    save_predictions=False,
    make_plots=True,
    overlay_limit=200,
):
    model.eval()

    split_dir = os.path.join(out_dir, split_name)
    os.makedirs(split_dir, exist_ok=True)

    overlay_dir = os.path.join(split_dir, "sample_overlays")
    os.makedirs(overlay_dir, exist_ok=True)

    instance_map_dir = os.path.join(split_dir, "instance_maps")
    if SAVE_ALL_INSTANCE_MAPS:
        os.makedirs(instance_map_dir, exist_ok=True)

    det_preds_by_class = {c: [] for c in CLASS_IDS}
    det_gt_by_class = {c: defaultdict(list) for c in CLASS_IDS}
    det_score_labels = {c: {"scores": [], "labels": []} for c in CLASS_IDS}
    det_thr_stats = {
        c: {"tp": 0, "fp": 0, "fn": 0, "matched_ious": []}
        for c in CLASS_IDS
    }

    seg_stats = {
        c: {"inter": 0, "union": 0, "pred_sum": 0, "gt_sum": 0}
        for c in CLASS_IDS
    }

    overlay_saved = 0
    pred_rows = []
    any_ground_truth_present = False

    jsonl_fp = None
    if save_predictions:
        jsonl_fp = open(os.path.join(split_dir, f"{split_name}_predictions.jsonl"), "w")

    with torch.no_grad():
        pbar = tqdm(loader, desc=f"Evaluating {split_name}")
        for images, targets, metas in pbar:
            batch_images = [img.to(device, non_blocking=True) for img in images]

            with autocast(device_type=AMP_DEVICE, enabled=AMP_ENABLED):
                outputs = model(batch_images)

            for image_cpu, target, meta, pred in zip(images, targets, metas, outputs):
                image_id = meta["stem"]
                H, W = image_cpu.shape[1], image_cpu.shape[2]

                gt_boxes = target["boxes"].cpu()
                gt_labels = target["labels"].cpu().numpy()
                gt_masks = target["masks"].cpu().numpy().astype(np.uint8)

                if len(gt_boxes) > 0:
                    any_ground_truth_present = True

                pred_boxes = pred["boxes"].detach().cpu()
                pred_labels = pred["labels"].detach().cpu().numpy()
                pred_scores = pred["scores"].detach().cpu().numpy()

                pred_masks_tensor = pred["masks"].detach().cpu()
                if len(pred_masks_tensor) > 0:
                    pred_masks = pred_masks_tensor[:, 0].numpy()
                else:
                    pred_masks = np.zeros((0, H, W), dtype=np.float32)

                pred_boxes_np = pred_boxes.numpy() if len(pred_boxes) > 0 else np.zeros((0, 4), dtype=np.float32)

                # -----------------------------------------
                # GT collection
                # -----------------------------------------
                for c in CLASS_IDS:
                    idxs = np.where(gt_labels == c)[0]
                    if len(idxs) > 0:
                        for b in gt_boxes[idxs].numpy():
                            det_gt_by_class[c][image_id].append(b.astype(np.float32))

                # -----------------------------------------
                # Prediction collection for AP
                # -----------------------------------------
                keep_eval = pred_scores >= AP_MIN_SCORE
                for box, label, score in zip(pred_boxes_np[keep_eval], pred_labels[keep_eval], pred_scores[keep_eval]):
                    if int(label) in CLASS_IDS:
                        det_preds_by_class[int(label)].append({
                            "image_id": image_id,
                            "score": float(score),
                            "box": box.astype(np.float32),
                        })

                # -----------------------------------------
                # Detection ROC/AUC/F1/IoU
                # -----------------------------------------
                for c in CLASS_IDS:
                    gt_idx = np.where(gt_labels == c)[0]
                    gt_boxes_c = gt_boxes[gt_idx].numpy() if len(gt_idx) > 0 else np.zeros((0, 4), dtype=np.float32)

                    pred_idx_all = np.where(pred_labels == c)[0]
                    if len(pred_idx_all) > 0:
                        order = np.argsort(pred_scores[pred_idx_all])[::-1]
                        pred_idx_all = pred_idx_all[order]

                    used_gt_for_roc = np.zeros(len(gt_boxes_c), dtype=bool)

                    for pi in pred_idx_all:
                        score = float(pred_scores[pi])
                        best_iou = 0.0
                        best_gi = -1

                        if len(gt_boxes_c) > 0:
                            ious = box_iou(
                                pred_boxes[pi:pi+1],
                                torch.tensor(gt_boxes_c, dtype=torch.float32)
                            ).squeeze(0).numpy()
                            best_gi = int(np.argmax(ious))
                            best_iou = float(ious[best_gi])

                        is_tp = int(best_gi >= 0 and best_iou >= MATCH_IOU_THRESH and not used_gt_for_roc[best_gi])
                        if is_tp:
                            used_gt_for_roc[best_gi] = True

                        det_score_labels[c]["scores"].append(score)
                        det_score_labels[c]["labels"].append(is_tp)

                    missed = int((~used_gt_for_roc).sum())
                    det_score_labels[c]["scores"].extend([0.0] * missed)
                    det_score_labels[c]["labels"].extend([1] * missed)

                    pred_idx_thr = [pi for pi in pred_idx_all if pred_scores[pi] >= SAVE_SCORE_THRESH]
                    used_gt_thr = np.zeros(len(gt_boxes_c), dtype=bool)

                    for pi in pred_idx_thr:
                        best_iou = 0.0
                        best_gi = -1

                        if len(gt_boxes_c) > 0:
                            ious = box_iou(
                                pred_boxes[pi:pi+1],
                                torch.tensor(gt_boxes_c, dtype=torch.float32)
                            ).squeeze(0).numpy()
                            best_gi = int(np.argmax(ious))
                            best_iou = float(ious[best_gi])

                        if best_gi >= 0 and best_iou >= MATCH_IOU_THRESH and not used_gt_thr[best_gi]:
                            used_gt_thr[best_gi] = True
                            det_thr_stats[c]["tp"] += 1
                            det_thr_stats[c]["matched_ious"].append(best_iou)
                        else:
                            det_thr_stats[c]["fp"] += 1

                    det_thr_stats[c]["fn"] += int((~used_gt_thr).sum())

                # -----------------------------------------
                # Segmentation IoU / Dice per class
                # -----------------------------------------
                for c in CLASS_IDS:
                    gt_idx = np.where(gt_labels == c)[0]
                    if len(gt_idx) > 0:
                        gt_union = np.any(gt_masks[gt_idx].astype(bool), axis=0)
                    else:
                        gt_union = np.zeros((H, W), dtype=bool)

                    pred_idx_seg = np.where((pred_labels == c) & (pred_scores >= SAVE_SCORE_THRESH))[0]
                    if len(pred_idx_seg) > 0:
                        pred_union = np.any((pred_masks[pred_idx_seg] >= 0.5), axis=0)
                    else:
                        pred_union = np.zeros((H, W), dtype=bool)

                    inter = np.logical_and(gt_union, pred_union).sum()
                    union = np.logical_or(gt_union, pred_union).sum()

                    seg_stats[c]["inter"] += int(inter)
                    seg_stats[c]["union"] += int(union)
                    seg_stats[c]["pred_sum"] += int(pred_union.sum())
                    seg_stats[c]["gt_sum"] += int(gt_union.sum())

                # -----------------------------------------
                # Save predictions
                # -----------------------------------------
                if save_predictions:
                    keep_save = pred_scores >= SAVE_SCORE_THRESH

                    save_boxes = pred_boxes_np[keep_save]
                    save_labels = pred_labels[keep_save]
                    save_scores = pred_scores[keep_save]
                    save_masks = (pred_masks[keep_save] >= 0.5).astype(np.uint8)

                    mask_rel_path = None
                    if SAVE_ALL_INSTANCE_MAPS:
                        instance_map = np.zeros((H, W), dtype=np.uint16)
                        for inst_id, mask in enumerate(save_masks, start=1):
                            instance_map[mask.astype(bool)] = inst_id

                        mask_rel_path = f"instance_maps/{image_id}.png"
                        Image.fromarray(instance_map, mode="I;16").save(
                            os.path.join(split_dir, mask_rel_path)
                        )

                    image_record = {
                        "image_id": image_id,
                        "image_path": meta["image_path"],
                        "height": int(H),
                        "width": int(W),
                        "has_ground_truth": bool(meta.get("has_gt", False)),
                        "instance_map_path": mask_rel_path,
                        "instances": []
                    }

                    for inst_id, (box, label, score) in enumerate(zip(save_boxes, save_labels, save_scores), start=1):
                        image_record["instances"].append({
                            "instance_id": int(inst_id),
                            "label_id": int(label),
                            "label_name": TRAIN_LABEL_TO_NAME.get(int(label), f"class_{label}"),
                            "score": float(score),
                            "bbox_xyxy": [float(v) for v in box.tolist()],
                        })

                        pred_rows.append({
                            "image_id": image_id,
                            "instance_id": int(inst_id),
                            "label_id": int(label),
                            "label_name": TRAIN_LABEL_TO_NAME.get(int(label), f"class_{label}"),
                            "score": float(score),
                            "x1": float(box[0]),
                            "y1": float(box[1]),
                            "x2": float(box[2]),
                            "y2": float(box[3]),
                        })

                    jsonl_fp.write(json.dumps(image_record) + "\n")

                    if overlay_saved < overlay_limit:
                        save_overlay_image(
                            image_cpu,
                            save_boxes,
                            save_labels,
                            save_scores,
                            save_masks,
                            os.path.join(overlay_dir, f"{image_id}.jpg")
                        )
                        overlay_saved += 1

    if jsonl_fp is not None:
        jsonl_fp.close()

    iou_thresholds = np.arange(0.50, 0.96, 0.05)

    det_rows = []
    seg_rows = []

    all_ap50 = []
    all_coco_map = []
    all_auc = []
    all_f1 = []
    all_iou = []

    for c in CLASS_IDS:
        class_name = TRAIN_LABEL_TO_NAME[c]

        ap50 = compute_ap_101pt(det_preds_by_class[c], det_gt_by_class[c], iou_thr=0.50)
        ap_coco_list = [
            compute_ap_101pt(det_preds_by_class[c], det_gt_by_class[c], iou_thr=float(t))
            for t in iou_thresholds
        ]
        coco_map = safe_mean(ap_coco_list)

        tp = det_thr_stats[c]["tp"]
        fp = det_thr_stats[c]["fp"]
        fn = det_thr_stats[c]["fn"]

        precision = tp / (tp + fp + 1e-12)
        recall = tp / (tp + fn + 1e-12)
        f1 = 2 * precision * recall / (precision + recall + 1e-12)
        mean_det_iou = safe_mean(det_thr_stats[c]["matched_ious"])

        scores_arr = np.array(det_score_labels[c]["scores"], dtype=np.float32)
        labels_arr = np.array(det_score_labels[c]["labels"], dtype=np.int32)

        auc = np.nan
        if len(scores_arr) > 0 and len(np.unique(labels_arr)) >= 2:
            try:
                auc = float(roc_auc_score(labels_arr, scores_arr))
                if make_plots:
                    fpr, tpr, _ = roc_curve(labels_arr, scores_arr)
                    plt.figure(figsize=(6, 5))
                    plt.plot(fpr, tpr, label=f"AUC = {auc:.4f}")
                    plt.plot([0, 1], [0, 1], linestyle="--")
                    plt.xlabel("False Positive Rate")
                    plt.ylabel("True Positive Rate")
                    plt.title(f"ROC - {split_name} - {class_name}")
                    plt.legend()
                    plt.grid(True)
                    plt.tight_layout()
                    plt.savefig(os.path.join(split_dir, f"roc_{class_name.replace(' ', '_')}.png"), dpi=150)
                    plt.close()
            except Exception:
                auc = np.nan

        det_rows.append({
            "class_id": c,
            "class_name": class_name,
            "num_gt_instances": int(sum(len(v) for v in det_gt_by_class[c].values())),
            "detection_iou": mean_det_iou,
            "ap50": ap50,
            "coco_map_50_95": coco_map,
            "precision_at_score_thresh": precision,
            "recall_at_score_thresh": recall,
            "f1_at_score_thresh": f1,
            "auc": auc,
            "tp": int(tp),
            "fp": int(fp),
            "fn": int(fn),
        })

        all_ap50.append(ap50)
        all_coco_map.append(coco_map)
        all_auc.append(auc)
        all_f1.append(f1)
        all_iou.append(mean_det_iou)

        inter = seg_stats[c]["inter"]
        union = seg_stats[c]["union"]
        pred_sum = seg_stats[c]["pred_sum"]
        gt_sum = seg_stats[c]["gt_sum"]

        seg_iou = inter / (union + 1e-12) if union > 0 else np.nan
        seg_dice = (2 * inter) / (pred_sum + gt_sum + 1e-12) if (pred_sum + gt_sum) > 0 else np.nan

        seg_rows.append({
            "class_id": c,
            "class_name": class_name,
            "intersection": int(inter),
            "union": int(union),
            "pred_pixels": int(pred_sum),
            "gt_pixels": int(gt_sum),
            "iou": seg_iou,
            "dice": seg_dice,
        })

    det_df = pd.DataFrame(det_rows)
    seg_df = pd.DataFrame(seg_rows)

    det_df.to_csv(os.path.join(split_dir, f"{split_name}_detection_per_class.csv"), index=False)
    seg_df.to_csv(os.path.join(split_dir, f"{split_name}_segmentation_per_class.csv"), index=False)

    if len(pred_rows) > 0:
        pd.DataFrame(pred_rows).to_csv(os.path.join(split_dir, f"{split_name}_pred_instances.csv"), index=False)

    overall_tp = int(det_df["tp"].sum()) if "tp" in det_df else 0
    overall_fp = int(det_df["fp"].sum()) if "fp" in det_df else 0
    overall_fn = int(det_df["fn"].sum()) if "fn" in det_df else 0

    overall_precision = overall_tp / (overall_tp + overall_fp + 1e-12)
    overall_recall = overall_tp / (overall_tp + overall_fn + 1e-12)
    overall_f1 = 2 * overall_precision * overall_recall / (overall_precision + overall_recall + 1e-12)

    summary = {
        "split": split_name,
        "has_ground_truth_anywhere": bool(any_ground_truth_present),

        # Detection
        "detection_iou": safe_mean(all_iou),
        "ap50": safe_mean(all_ap50),
        "map50": safe_mean(all_ap50),
        "coco_map_50_95": safe_mean(all_coco_map),
        "macro_auc": safe_mean(all_auc),
        "macro_f1": safe_mean(all_f1),
        "overall_precision_at_score_thresh": overall_precision,
        "overall_recall_at_score_thresh": overall_recall,
        "overall_f1_at_score_thresh": overall_f1,

        # Segmentation
        "segmentation_iou": safe_mean(seg_df["iou"].tolist()),
        "segmentation_miou": safe_mean(seg_df["iou"].tolist()),
        "segmentation_dice": safe_mean(seg_df["dice"].tolist()),

        # Counts
        "num_images": int(len(loader.dataset)),
        "num_pred_rows": int(len(pred_rows)),
    }

    with open(os.path.join(split_dir, f"{split_name}_summary.json"), "w") as f:
        json.dump(summary, f, indent=2)

    print(f"\n===== {split_name.upper()} SUMMARY =====")
    for k, v in summary.items():
        print(f"{k}: {v}")

    return summary, det_df, seg_df

# ============================================================
# 16. History plots
# ============================================================
def save_history_plots(history, out_dir):
    epochs = list(range(1, len(history["train_loss"]) + 1))

    def clean(vals):
        return [np.nan if v is None else v for v in vals]

    plt.figure(figsize=(8, 5))
    plt.plot(epochs, history["train_loss"], marker="o")
    plt.xlabel("Epoch")
    plt.ylabel("Train Loss")
    plt.title("Training Loss")
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(os.path.join(out_dir, "train_loss.png"), dpi=150)
    plt.close()

    plt.figure(figsize=(8, 5))
    plt.plot(epochs, clean(history["quick_val_map50"]), marker="o", label="Quick Val mAP@0.5")
    plt.plot(epochs, clean(history["quick_val_coco_map"]), marker="o", label="Quick Val COCO mAP")
    plt.plot(epochs, clean(history["quick_val_seg_miou"]), marker="o", label="Quick Val Seg mIoU")
    plt.plot(epochs, clean(history["quick_val_macro_f1"]), marker="o", label="Quick Val Macro F1")
    plt.xlabel("Epoch")
    plt.ylabel("Metric")
    plt.title("Quick Validation Metrics")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(os.path.join(out_dir, "quick_val_metrics.png"), dpi=150)
    plt.close()

# ============================================================
# 17. Train
# ============================================================
history = {
    "train_loss": [],
    "loss_parts": [],
    "quick_val_map50": [],
    "quick_val_coco_map": [],
    "quick_val_seg_miou": [],
    "quick_val_macro_f1": [],
}

best_metric = -1.0
best_epoch = -1
epochs_without_improvement = 0

best_path = os.path.join(OUT_DIR, "maskrcnn_best.pth")
last_path = os.path.join(OUT_DIR, "maskrcnn_last.pth")

for epoch in range(1, MAX_EPOCHS + 1):
    print(f"\n========== Epoch {epoch}/{MAX_EPOCHS} ==========")

    train_loss, loss_parts = train_one_epoch(
        model=model,
        loader=train_loader,
        optimizer=optimizer,
        device=DEVICE,
        epoch=epoch,
        scaler=SCALER,
        accum_steps=GRAD_ACCUM_STEPS,
    )

    history["train_loss"].append(train_loss)
    history["loss_parts"].append(loss_parts)

    should_validate = (epoch % EVAL_EVERY == 0) or (epoch == MAX_EPOCHS)

    if should_validate:
        quick_summary, _, _ = evaluate_split(
            model=model,
            loader=val_subset_loader,
            device=DEVICE,
            split_name=f"quick_val_epoch_{epoch}",
            out_dir=OUT_DIR,
            save_predictions=False,
            make_plots=False,
            overlay_limit=0,
        )

        history["quick_val_map50"].append(quick_summary["map50"])
        history["quick_val_coco_map"].append(quick_summary["coco_map_50_95"])
        history["quick_val_seg_miou"].append(quick_summary["segmentation_miou"])
        history["quick_val_macro_f1"].append(quick_summary["macro_f1"])

        current_metric = (
            (0.45 * (0 if np.isnan(quick_summary["coco_map_50_95"]) else quick_summary["coco_map_50_95"])) +
            (0.35 * (0 if np.isnan(quick_summary["segmentation_miou"]) else quick_summary["segmentation_miou"])) +
            (0.20 * (0 if np.isnan(quick_summary["macro_f1"]) else quick_summary["macro_f1"]))
        )

        print("Train loss:", train_loss)
        print("Loss parts:", loss_parts)
        print("Quick val summary:", quick_summary)
        print("Combined metric:", current_metric)

        if current_metric > best_metric:
            best_metric = current_metric
            best_epoch = epoch
            epochs_without_improvement = 0

            torch.save({
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "scheduler_state_dict": scheduler.state_dict(),
                "history": history,
                "best_metric": best_metric,
                "config": {
                    "top5_orig_ids": TOP5_ORIG_IDS,
                    "num_classes": NUM_CLASSES,
                    "lr": LR,
                    "max_epochs": MAX_EPOCHS,
                    "min_size": MIN_SIZE,
                    "max_size": MAX_SIZE,
                    "box_detections_per_img": BOX_DETECTIONS_PER_IMG,
                    "trainable_backbone_layers": TRAINABLE_BACKBONE_LAYERS,
                }
            }, best_path)

            print("Saved new BEST model:", best_path)
        else:
            epochs_without_improvement += 1
            print(f"No improvement. Patience: {epochs_without_improvement}/{PATIENCE}")

    else:
        history["quick_val_map50"].append(None)
        history["quick_val_coco_map"].append(None)
        history["quick_val_seg_miou"].append(None)
        history["quick_val_macro_f1"].append(None)

        print("Train loss:", train_loss)
        print("Loss parts:", loss_parts)
        print("Quick validation skipped.")

    torch.save({
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),
        "history": history,
        "best_metric": best_metric,
        "config": {
            "top5_orig_ids": TOP5_ORIG_IDS,
            "num_classes": NUM_CLASSES,
            "lr": LR,
            "max_epochs": MAX_EPOCHS,
            "min_size": MIN_SIZE,
            "max_size": MAX_SIZE,
            "box_detections_per_img": BOX_DETECTIONS_PER_IMG,
            "trainable_backbone_layers": TRAINABLE_BACKBONE_LAYERS,
        }
    }, last_path)

    with open(os.path.join(OUT_DIR, "history.json"), "w") as f:
        json.dump(history, f, indent=2)

    save_history_plots(history, OUT_DIR)
    scheduler.step()

    if epochs_without_improvement >= PATIENCE:
        print(f"Early stopping triggered at epoch {epoch}. Best epoch = {best_epoch}")
        break

print("\nTraining finished.")
print("Best epoch:", best_epoch)
print("Best metric:", best_metric)

# ============================================================
# 18. Load best model
# ============================================================
if os.path.exists(best_path):
    ckpt = torch.load(best_path, map_location=DEVICE)
    model.load_state_dict(ckpt["model_state_dict"])
    print("Loaded BEST checkpoint from:", best_path)
else:
    ckpt = torch.load(last_path, map_location=DEVICE)
    model.load_state_dict(ckpt["model_state_dict"])
    print("BEST checkpoint missing; loaded LAST checkpoint instead.")

# ============================================================
# 19. Final FULL evaluation on validation
# ============================================================
final_val_summary, final_val_det_df, final_val_seg_df = evaluate_split(
    model=model,
    loader=val_loader,
    device=DEVICE,
    split_name="final_val",
    out_dir=OUT_DIR,
    save_predictions=True,
    make_plots=True,
    overlay_limit=SAVE_SAMPLE_OVERLAYS,
)

# ============================================================
# 20. Final test evaluation / inference
# ============================================================
final_test_summary = None

if test_loader is not None:
    final_test_summary, final_test_det_df, final_test_seg_df = evaluate_split(
        model=model,
        loader=test_loader,
        device=DEVICE,
        split_name="final_test",
        out_dir=OUT_DIR,
        save_predictions=True,
        make_plots=True,
        overlay_limit=SAVE_SAMPLE_OVERLAYS,
    )
else:
    print("Test loader is None, so final test inference/evaluation is skipped.")

# ============================================================
# 21. Save final compact report
# ============================================================
final_report_rows = [
    {
        "split": "validation",
        **final_val_summary
    }
]

if final_test_summary is not None:
    final_report_rows.append({
        "split": "test",
        **final_test_summary
    })

final_report_df = pd.DataFrame(final_report_rows)
final_report_csv = os.path.join(OUT_DIR, "final_summary_report.csv")
final_report_df.to_csv(final_report_csv, index=False)

print("\nSaved final summary report to:", final_report_csv)

# ============================================================
# 22. Cleanup
# ============================================================
gc.collect()
torch.cuda.empty_cache()

print("\nDone.")
print("Output directory:", OUT_DIR)
print("Top-level files/folders:")
for x in sorted(os.listdir(OUT_DIR)):
    print(" -", x)